In [1]:
import google.protobuf
import grpc_status
print("protobuf OK", google.protobuf.__version__)

protobuf OK 6.33.4


In [2]:
from pyspark.sql import SparkSession

from datachecker.engine import ValidationEngine
from datachecker.schema_loader import YamlFileSchemaLoader
from datachecker.plans import PanderaPlanCompiler
from datachecker.batch_validators import PanderaBatchValidator
from datachecker.batch_sources import SparkDataFrameBatchSource
from datachecker.sinks import PrintSink

spark = (
    SparkSession.builder
    .remote("sc://127.0.0.1:15002?connectTimeoutMs=5000")
    .appName("dq-pandera-demo")
    .getOrCreate()
)

In [3]:
df = spark.read.format("delta").load("s3a://lake/delta_users")

In [4]:
engine = ValidationEngine(
    schema_loader=YamlFileSchemaLoader("../schemas"),
    compiler=PanderaPlanCompiler(),
    source=SparkDataFrameBatchSource(df),
    batch_validator=PanderaBatchValidator(),
    sink=PrintSink(),
)


In [5]:
engine.run("user")

{
  "event": "start",
  "schema": "user"
}
{
  "event": "chunk",
  "mode": "batch",
  "schema": "user",
  "batch_no": 1,
  "batch_size": null,
  "report": {
    "schema_name": "user",
    "total": null,
    "valid": null,
    "invalid": 0,
    "row_errors": [],
    "sample_failures": [],
    "details": {
      "validator": "pandera",
      "status": "PASS"
    },
    "ok": true
  }
}
{
  "event": "summary",
  "mode": "batch",
  "schema": "user",
  "batches": 1,
  "invalid_total": 0
}
{
  "event": "end",
  "schema": "user"
}


In [ ]:
spark.stop()

In [ ]:
PanderaPlanCompiler()